In [ ]:
import polars as pl
from abc import ABC, abstractmethod
from dataclasses import dataclass
from typing import Iterable, Optional, Callable, List, Any

In [ ]:
class Label(dict):
    def matches(self, **kwargs):
        for k, v in kwargs.items():
            if k not in self.keys():
                raise ValueError(f"Trying to match on unknown namespace {k}")
            elif self[k] != v:
                return False
        return True
    
    def drop(self, *keys):
        new_mapping = {k:v for k, v in self.items() if k not in keys}
        return Label(**new_mapping)

In [ ]:
class PomapNode(ABC):
    @property
    @abstractmethod
    def children(self) -> Iterable["PomapNode"]:
        """Return iterable of child nodes."""
        ...

In [ ]:
@dataclass
class Lift(PomapNode):
    child: PomapNode
    atomics: Iterable[pl.DataType]
    name: Optional[str] = None

    def __post_init__(self):
        if self.name is None:
            self.name = f"Lift: {self.atomics}"
        self.atomics = set(self.atomics)

    @property
    def children(self) -> Iterable["PomapNode"]:
        return [self.child]


In [ ]:
@dataclass
class Leaf(PomapNode):
    label: str

    @abstractmethod
    def fit(self):
        ...
    @abstractmethod
    def predict(self):
        ...

    @property
    def children(self):
        return []


### Define interpreter

In [ ]:
### Now define the different behaviours that we want to apply to a tree

def _collect_labels(expr: PomapNode) -> List[Label]:
    match expr:
        case leaf if isinstance(leaf, Leaf):
            return [Label(leaf=leaf.label)]
        case Lift(child, atomics, name):
            labels = []
            for child_label in _collect_labels(child):
                for atomic in atomics:
                    labels.append(
                        Label({**child_label, name: atomic})
                    )
            return labels
    return []


def _collect_leaves(node: PomapNode) -> List[Label]:
    match node.children:
        case []:
            yield node
        case children:
            for child in children:
                yield from _collect_leaves(child)


### Tests

In [ ]:
from scipy import stats

In [ ]:
@dataclass
class LinearRegression(Leaf):
    x_column: str
    y_column: str
    intercept: float = None
    beta: float = None
    
    def fit(df: pl.DataFrame) -> None:
        x = df[self.x_column]
        y = df[self.y_column]

        lr = stats.linregress(x, y)
        self.intercept = lr[0]
        self.beta = lr[1]

    def predict(df: pl.DataFrame) -> pl.DataFrame:
        df = df.with_columns(predictions=pl.col(self.x_column)*self.beta + self.intercept)

        return df
        

In [ ]:
class BasicLift(Lift):
     
    def train_mask_for_label(self, label) -> pl.Expr:
        return pl.col('column') == pl.lit(label)

In [ ]:
model = LinearRegression(label='lr', x_column='x', y_column='y')
lifted = BasicLift(model, atomics=['a', 'b', 'c'])

In [ ]:
_collect_labels( model)

In [ ]:
_collect_labels(lifted)

In [ ]:
list(_collect_leaves(model))

In [ ]:
list(_collect_leaves(lifted))